In [2]:
# Load the data and the embedding model 

%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

import pandas as pd
from sentence_transformers import SentenceTransformer
from src.evaluation import brand_of, split_by_brand, evaluate

df = pd.read_csv("../data/processed/transactions_clean.csv")
df["brand"] = df["merchant"].apply(brand_of)
train, test = split_by_brand(df)

encoder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print(f"Train: {len(train)} | Test: {len(test)}")
print(f"Embedding size: {encoder.encode(['test']).shape[1]}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Train: 3969 | Test: 991
Embedding size: 384


In [3]:
# Does the model understand our labels ? 

from sentence_transformers.util import cos_sim

labels = [
    "CB PATHE CINEMAS",
    "CB BURGER KING",
    "CB UBER TRIP",
    "CB NETFLIX COM",
    "CB CARREFOUR MARKET",
    "CB ZARA",
]
concepts = ["cinema", "restaurant", "taxi", "streaming", "supermarket", "clothes"]
concept_vectors = encoder.encode(concepts)

for name, texts in [("UPPERCASE", labels), ("lowercase", [label.lower() for label in labels])]:
    similarities = cos_sim(encoder.encode(texts), concept_vectors)
    table = pd.DataFrame(similarities.numpy(), index=texts, columns=concepts).round(2)
    print(f"--- {name} ---")
    print(table, "\n")

--- UPPERCASE ---
                     cinema  restaurant  taxi  streaming  supermarket  clothes
CB PATHE CINEMAS       0.39        0.24  0.24       0.17         0.24     0.26
CB BURGER KING         0.29        0.32  0.25       0.03         0.30     0.11
CB UBER TRIP           0.37        0.19  0.37       0.28         0.17     0.23
CB NETFLIX COM         0.23        0.16  0.16       0.42         0.16     0.07
CB CARREFOUR MARKET    0.29        0.27  0.22       0.17         0.49     0.17
CB ZARA                0.52        0.26  0.30       0.29         0.28     0.34 

--- lowercase ---
                     cinema  restaurant  taxi  streaming  supermarket  clothes
cb pathe cinemas       0.67        0.27  0.22       0.37         0.26     0.18
cb burger king         0.18        0.36  0.17       0.06         0.35     0.10
cb uber trip           0.23        0.17  0.31       0.25         0.19     0.13
cb netflix com         0.37        0.13  0.11       0.64         0.21    -0.03
cb carrefour m

In [4]:
# Embed the train and test labels 

def embed(texts):
    return encoder.encode(
        [text.lower() for text in texts],
        batch_size=64,
        show_progress_bar=True,
    )

X_train = embed(train["clean_label"])
X_test = embed(test["clean_label"])

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

X_train: (3969, 384) | X_test: (991, 384)


In [5]:
# Train and evaluate 

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

embedding_model = LogisticRegression(max_iter=1000)
embedding_model.fit(X_train, train["category"])
embedding_pred = embedding_model.predict(X_test)

evaluate(test["category"], embedding_pred, "Embeddings (synthetic test)")
print()
print(classification_report(test["category"], embedding_pred, digits=3))

Embeddings (synthetic test)      accuracy: 46.8% | macro F1: 47.8%

                     precision    recall  f1-score   support

          Bank Fees      0.977     0.694     0.811        62
    Cash Withdrawal      0.953     1.000     0.976       102
          Groceries      0.386     0.500     0.436       102
             Health      0.659     0.636     0.647        88
Housing & Utilities      0.495     0.739     0.593        69
             Income      0.547     0.547     0.547        53
   Leisure & Travel      0.492     0.326     0.392        92
        Restaurants      0.318     0.143     0.197        98
           Shopping      0.120     0.120     0.120       100
      Subscriptions      0.260     0.281     0.270        89
          Transfers      0.431     0.609     0.505        46
          Transport      0.228     0.256     0.241        90

           accuracy                          0.468       991
          macro avg      0.489     0.488     0.478       991
       weighted

In [6]:
# Comparison of the 3 models

from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from src.evaluation import cross_val_by_brand

texts = train["clean_label"]
y = train["category"]
X_emb = normalize(X_train)


def make_tfidf():
    return TfidfVectorizer(ngram_range=(1, 2), token_pattern=r"(?u)\b\w+\b")


def tfidf_fit_predict(train_idx, val_idx):
    vectorizer = make_tfidf()
    X_tr = vectorizer.fit_transform(texts.iloc[train_idx])
    X_val = vectorizer.transform(texts.iloc[val_idx])
    model = LogisticRegression(max_iter=1000).fit(X_tr, y.iloc[train_idx])
    return model.predict(X_val)


def embeddings_fit_predict(train_idx, val_idx):
    model = LogisticRegression(max_iter=1000).fit(X_emb[train_idx], y.iloc[train_idx])
    return model.predict(X_emb[val_idx])


def hybrid_fit_predict(train_idx, val_idx):
    vectorizer = make_tfidf()
    X_tr = hstack([vectorizer.fit_transform(texts.iloc[train_idx]), csr_matrix(X_emb[train_idx])])
    X_val = hstack([vectorizer.transform(texts.iloc[val_idx]), csr_matrix(X_emb[val_idx])])
    model = LogisticRegression(max_iter=1000).fit(X_tr, y.iloc[train_idx])
    return model.predict(X_val)


for name, fit_predict in [
    ("TF-IDF", tfidf_fit_predict),
    ("Embeddings", embeddings_fit_predict),
    ("Hybrid", hybrid_fit_predict),
]:
    mean, std = cross_val_by_brand(fit_predict, train)
    print(f"{name:<12} macro F1: {mean:.1%} ± {std:.1%}")

TF-IDF       macro F1: 50.6% ± 2.9%
Embeddings   macro F1: 51.6% ± 2.6%
Hybrid       macro F1: 58.0% ± 2.5%


In [7]:
# Try several values of C 

def make_hybrid_fit_predict(C):
    def fit_predict(train_idx, val_idx):
        vectorizer = make_tfidf()
        X_tr = hstack([vectorizer.fit_transform(texts.iloc[train_idx]), csr_matrix(X_emb[train_idx])])
        X_val = hstack([vectorizer.transform(texts.iloc[val_idx]), csr_matrix(X_emb[val_idx])])
        model = LogisticRegression(C=C, max_iter=1000).fit(X_tr, y.iloc[train_idx])
        return model.predict(X_val)
    return fit_predict


for C in [0.3, 1, 3, 10, 30]:
    mean, std = cross_val_by_brand(make_hybrid_fit_predict(C), train)
    print(f"C = {C:<5} macro F1: {mean:.1%} ± {std:.1%}")

C = 0.3   macro F1: 56.6% ± 2.8%
C = 1     macro F1: 58.0% ± 2.5%
C = 3     macro F1: 58.2% ± 2.5%
C = 10    macro F1: 58.5% ± 2.8%
C = 30    macro F1: 58.7% ± 2.6%


In [8]:
# The hybrid on the synthetic test set 

vectorizer = make_tfidf()
X_hybrid_train = hstack([vectorizer.fit_transform(train["clean_label"]), csr_matrix(X_emb)])
X_hybrid_test = hstack([vectorizer.transform(test["clean_label"]), csr_matrix(normalize(X_test))])

hybrid = LogisticRegression(C=1, max_iter=1000).fit(X_hybrid_train, train["category"])
hybrid_pred = hybrid.predict(X_hybrid_test)

evaluate(test["category"], hybrid_pred, "Hybrid (synthetic test)")
print()
print(classification_report(test["category"], hybrid_pred, digits=3))

Hybrid (synthetic test)          accuracy: 54.7% | macro F1: 57.4%

                     precision    recall  f1-score   support

          Bank Fees      1.000     0.839     0.912        62
    Cash Withdrawal      1.000     1.000     1.000       102
          Groceries      0.490     0.480     0.485       102
             Health      0.791     0.773     0.782        88
Housing & Utilities      0.432     0.826     0.567        69
             Income      0.779     1.000     0.876        53
   Leisure & Travel      0.818     0.293     0.432        92
        Restaurants      0.509     0.276     0.358        98
           Shopping      0.174     0.330     0.228       100
      Subscriptions      0.322     0.213     0.257        89
          Transfers      0.721     0.674     0.697        46
          Transport      0.329     0.267     0.294        90

           accuracy                          0.547       991
          macro avg      0.614     0.581     0.574       991
       weighted

In [9]:
# The hybrid on the realistic set 

from src.preprocessing import clean_label

realistic = pd.read_csv("../data/test_realistic.csv")
realistic["clean_label"] = realistic["label"].apply(clean_label)

full_vectorizer = make_tfidf()
X_all = hstack([
    full_vectorizer.fit_transform(df["clean_label"]),
    csr_matrix(normalize(embed(df["clean_label"]))),
])
X_real = hstack([
    full_vectorizer.transform(realistic["clean_label"]),
    csr_matrix(normalize(embed(realistic["clean_label"]))),
])

full_hybrid = LogisticRegression(C=1, max_iter=1000).fit(X_all, df["category"])
realistic["predicted"] = full_hybrid.predict(X_real)
realistic["confidence"] = full_hybrid.predict_proba(X_real).max(axis=1)

evaluate(realistic["category"], realistic["predicted"], "Hybrid (realistic set)")

errors = realistic[realistic["predicted"] != realistic["category"]]
print(f"\n{len(errors)} errors out of {len(realistic)}\n")
errors[["clean_label", "category", "predicted", "confidence"]].sort_values("confidence")

Batches:   0%|          | 0/78 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Hybrid (realistic set)           accuracy: 71.7% | macro F1: 71.0%

17 errors out of 60



,clean_label,category,predicted,confidence
14,FACTURE CARTE DU Q PARK FRANCE CARTE 4974XXXXX...,Transport,Restaurants,0.285449
12,PRLV SEPA BIP GO ECH,Transport,Housing & Utilities,0.310684
38,CB SEETICKETS,Leisure & Travel,Shopping,0.320247
7,CB SUMUP LA TRATTORIA,Restaurants,Transport,0.321138
37,PRLV SEPA L ORANGE BLEUE,Leisure & Travel,Housing & Utilities,0.340141
21,FACTURE CARTE DU BOULANGER LA DEFENSE CARTE 49...,Shopping,Restaurants,0.344302
0,CARTE BIOCBON PARIS 11,Groceries,Health,0.346163
23,CB TEMU COM,Shopping,Subscriptions,0.367938
28,PRLV SEPA LEOCARE ASSURANCE,Housing & Utilities,Health,0.404453
45,VIR INST VERS M DUPONT THOMAS,Transfers,Housing & Utilities,0.410705
